# EcoEye — Bird Image Scraper v1: Initial Prototype

**Iteration:** First working prototype — scrapes bird images from the **eBird Macaulay Library** using Selenium.  
**Source:** `media.ebird.org` — research-grade bird observation photos  
**Limitation:** Single page only, no pagination, no standardized filenames.

> ⚠️ This is the initial development prototype. Iteration history:
> - `v1` → `Web Scraping.ipynb` ← **(you are here)**
> - `v2` → `WebScraping_new.ipynb` — adds extension filtering
> - `v3` → `Web Scraping with names and datetime(Final).ipynb` — adds species-tagged filenames + pagination
> - `v4` → `Final Scraping (Code).ipynb` — production version with configurable inputs

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import requests
import os
import time

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────────
TAXON_CODE        = "afecuc1"          # eBird taxon code — find yours at https://ebird.org/explore
CHROMEDRIVER_PATH = "chromedriver.exe" # Place chromedriver.exe in this folder, or provide full path
SPECIES_NAME      = "African_Emerald_Cuckoo"

URL = f"https://media.ebird.org/catalog?taxonCode={TAXON_CODE}&mediaType=photo"

SAVE_FOLDER = os.path.join(os.getcwd(), "downloaded_images", SPECIES_NAME)
os.makedirs(SAVE_FOLDER, exist_ok=True)
print(f"Saving images to: {SAVE_FOLDER}")

In [ ]:
# Launch browser and scrape image URLs
driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH))
driver.get(URL)
time.sleep(5)  # Wait for dynamic content to load

image_elements = driver.find_elements(By.TAG_NAME, 'img')
jpg_png_images = [
    img.get_attribute('src')
    for img in image_elements
    if img.get_attribute('src') and img.get_attribute('src').lower().endswith(('.png', '.jpg', '.jpeg'))
]

driver.quit()
print(f"Found {len(jpg_png_images)} images")

In [ ]:
# Download and save images
for image_url in jpg_png_images:
    response = requests.get(image_url, stream=True)
    if response.status_code == 200:
        image_name = image_url.split('/')[-1]
        save_path = os.path.join(SAVE_FOLDER, image_name)
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded: {image_name}")
    else:
        print(f"Failed ({response.status_code}): {image_url}")